# MCP

**Model Context Protocol (MCP)** is an open protocol that standardizes how applications provide tools and context to language models. LangGraph agents can use tools defined on MCP servers through the langchain-mcp-adapters library.

![LangGraph Mail Agent Overview](image.png)

Install the `langchain-mcp-adapters` library to use MCP tools in LangGraph:

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_aws langchain_core langgraph
%pip install --upgrade python-dotenv
%pip install langchain-mcp-adapters 

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] =os.getenv('AWS_DEFAULT_REGION')

Here, we'll use [LangSmith](https://docs.smith.langchain.com/) for [tracing](https://docs.smith.langchain.com/concepts/tracing).

We'll log to a project, `langchain-academy`.

# Langsmith Setup

1. Create a LangSmith account at [smith.langchain.com](https://smith.langchain.com/)
2. Go to Tracing Projects
3. Click on New Project and select the `langgraph` project
4. Generate an API key for the project
5. Copy the `LANGSMITH_API_KEY` and paste it into your `.env` file

In [ ]:
os.environ["LANGSMITH_API_KEY"] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langgraph"

## Use MCP tool

The `langchain-mcp-adapters` package enables agents to use tools defined across one or more MCP servers.

To start MCP Server locally navigate to the `06_MCP/servers` folder and run:

```bash
python server.py
```

This will start an MCP server with multiple tools: `get_weather`, `add`, and `multiply`.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_aws import ChatBedrock
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# Initialize the model
llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    temperature=1,
)

# Set up MCP client
client = MultiServerMCPClient(
    {
        "server": {
            # make sure you start your server on port 8000
            "url": "http://127.0.0.1:8000/mcp/",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()

# Bind tools to model
model_with_tools = llm.bind_tools(tools)

# Create ToolNode
tool_node = ToolNode(tools)

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

# Define call_model function
async def call_model(state: MessagesState):
    messages = state["messages"]
    response = await model_with_tools.ainvoke(messages)
    return {"messages": [response]}

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "call_model")
builder.add_conditional_edges(
    "call_model",
    should_continue,
)
builder.add_edge("tools", "call_model")

# Compile the graph
graph = builder.compile()

# Test the graph
math_response = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)
weather_response = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
)

In [ ]:
print("Math Response:", math_response["messages"][-1].content)

In [ ]:
print("Weather Response:", weather_response["messages"][-1].content)